In [ ]:
## install module

!pip install -q google-generativeai langsmith

import os
import google.generativeai as genai
from google.colab import userdata
from langsmith import traceable

### setting up API key
genai.configure(
    api_key=userdata.get("GOOGLE_API_KEY")
)

### invoke gemini model

model = genai.GenerativeModel("gemini-2.5-flash")
chat = model.start_chat()

# langsmith
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "devops-chatbot"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"

#### traceable function

### take user input
user_input = input("please describe your devops requirement:")

@traceable
def generate_response(user_input):
  response = chat.send_message(f"You are a devops expert. The user says he needs help with: {user_input}. Based on the user query please provide precise and accurate answer")
  return response.text


## call LLM to generate response based on user input

result = generate_response(f"You are a devops expert. The user says he needs help with: {user_input}. Based on the user query please provide precise and accurate answer")
print("AI generated response")
print(result)

please describe your devops requirement:devops in 1 line
AI generated response
DevOps is a methodology that unifies software development and IT operations through collaboration, automation, and continuous feedback to deliver value rapidly and reliably.


In [13]:
# Install gradio
!pip install -q gradio

In [14]:
import gradio as gr

In [10]:
import os
import requests
import google.generativeai as genai
from google.colab import userdata


### setting up API key
genai.configure(
    api_key=userdata.get("GOOGLE_API_KEY")
)

model = genai.GenerativeModel("gemini-2.5-flash")

PROMETHEUS_URL = "http://136.115.103.194/"

### user request

user_question = input("Ask some monitoring question")

### generate promql

promql_prompt = f"""
Convert the following monitoring request into a promql query.

Request: {user_question}

Only return the PromQL query, nothing else.
"""

promql_response = model.generate_content(promql_prompt)

query = promql_response.text.strip()
# Remove markdown formatting from the PromQL query
if query.startswith('```promql') and query.endswith('```'):
    query = query[len('```promql\n'):-len('\n```')].strip()

print(f"PromQL query: \n{query}")

### fetch metrics from prometheus endpoint

response = requests.get(f"{PROMETHEUS_URL}/api/v1/query", params={"query": query})

metrics = response.json()

### AI analysis

analysis_prompt = f"""
You are an expert in Kubernetes & Observability

Analyze these prometheus metrics.

User Question:
{user_question}

PromQL Query:
{query}

Metrics:
{metrics}

Provide:
- summary
- root cause analysis
- remediation steps
- resolution summary

Please keep the output concise in 5-6 lines only

"""

analysis_response = model.generate_content(analysis_prompt)

print(f"AI Analysis: \n{analysis_response.text}")

Ask some monitoring questionis there any criticial observation on any of the nodes performance
PromQL query: 
(
  avg by (instance, job) (1 - rate(node_cpu_seconds_total{mode="idle"}[5m])) > 0.9
)
or
(
  node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes < 0.1
)
or
(
  max by (instance, job) (node_filesystem_avail_bytes{mountpoint="/", fstype!="rootfs"} / node_filesystem_size_bytes{mountpoint="/", fstype!="rootfs"} < 0.1)
)
AI Analysis: 
**Summary:** No critical performance issues detected on any node.
**Root Cause Analysis:** The PromQL query for high CPU (>90%), low memory (<10%), or low disk (<10%) returned an empty result, indicating no breaches.
**Remediation Steps:** No immediate action is required as all nodes are operating within healthy parameters. Continue routine monitoring.
**Resolution Summary:** Node performance is currently optimal, with no critical resource utilization thresholds breached.


### Gradio Interface for Monitoring Questions

In [ ]:
def monitoring_interface(user_question):
    promql_prompt = f"""
Convert the following monitoring request into a promql query.

Request: {user_question}

Only return the PromQL query, nothing else.
"""

    promql_response = model.generate_content(promql_prompt)
    query = promql_response.text.strip()
    # Remove markdown formatting from the PromQL query
    if query.startswith('```promql') and query.endswith('```'):
        query = query[len('```promql\n'):-len('\n```')].strip()

    try:
        response = requests.get(f"{PROMETHEUS_URL}/api/v1/query", params={"query": query})
        response.raise_for_status() # Raise an exception for HTTP errors
        metrics = response.json()
    except requests.exceptions.RequestException as e:
        return (f"Error generating PromQL query for '{user_question}':\n{query}",
                f"Error fetching metrics: {e}",
                "AI Analysis: Could not fetch metrics due to an error.")

    analysis_prompt = f"""
You are an expert in Kubernetes & Observability

Analyze these prometheus metrics.

User Question:
{user_question}

PromQL Query:
{query}

Metrics:
{metrics}

Provide:
- summary
- root cause analysis
- remediation steps
- resolution summary

Please keep the output concise in 5-6 lines only

"""

    analysis_response = model.generate_content(analysis_prompt)

    return (
        f"PromQL Query:\n```promql\n{query}\n```",
        f"Prometheus Metrics:\n```json\n{metrics}\n```",
        f"AI Analysis:\n{analysis_response.text}"
    )

# Create the Gradio interface
iface = gr.Interface(
    fn=monitoring_interface,
    inputs=gr.Textbox(lines=2, placeholder="Ask a monitoring question (e.g., 'What is the CPU usage of all pods?')"),
    outputs=[
        gr.Markdown(label="Generated PromQL Query"),
        gr.Markdown(label="Fetched Prometheus Metrics"),
        gr.Markdown(label="AI Analysis")
    ],
    title="Kubernetes & Observability Monitoring Assistant",
    description="Ask questions about your Kubernetes cluster's performance and get PromQL queries, metrics, and AI-driven analysis."
)

# Launch the Gradio app
iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ac97c37a253a468e41.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
